# Воспроизведение прогнозов платформы «Экспертиза национального проекта „Семья“»

Новая реализация 1.2.0. Для моделей показателей необходим NumPy; в первой ячейке установлена проверенная версия. Гармоника и квазипериодический GP оценивают изгибы по исходным наблюдениям; период 12 месяцев задан, а не доказан. Старые сохранённые результаты 1.1.1 повторяются отдельной реализацией. Загрузите JSON, скачанный с прогнозной страницы. Код ниже самодостаточен и не обращается к ЕМИСС; он воспроизводит именно сохранённый вход. Никакой ряд не переобозначается как наблюдение. Исходники соответствуют текущей версии пакета; при смене версии модели нужен новый блокнот.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.3.5"], check=True)


In [ ]:
from pathlib import Path
import json, subprocess, sys
FILES = {'demography/cohort.py': '"""Monthly cohort-component engine with an explicit 100+ open age group.\n\nSingle-year baseline cells are disaggregated into twelve equally-sized birth-month\nsubcohorts (an assumption, not new observed information). Residents age by exactly\none month. Births, deaths, and net migration reconcile to the stock identity.\nInputs with missing cells or impossible emigration are rejected, not repaired.\n"""\nfrom __future__ import annotations\nimport math\nfrom datetime import date\n\nVERSION = \'cohort-monthly/1.1.1\'\nSEXES = (\'male\',\'female\')\nN_AGE = 101\nEND = \'2030-12-31\'\n\n\ndef normalized(values):\n    if not isinstance(values,(list,tuple)) or any(not isinstance(x,(int,float)) or isinstance(x,bool) for x in values):\n        raise ValueError(\'Профиль должен содержать только числа.\')\n    a = [float(x) for x in values]\n    if not all(math.isfinite(x) and x >= 0 for x in a) or not math.isfinite(sum(a)) or sum(a) <= 0:\n        raise ValueError(\'Возрастной профиль должен быть конечным, неотрицательным и ненулевым.\')\n    total=sum(a)\n    return [x/total for x in a]\n\n\ndef fertility_profile(mean_age=28., sd=6.):\n    return normalized([math.exp(-.5*((a+.5-mean_age)/sd)**2) if 15 <= a <= 49 else 0 for a in range(N_AGE)])\n\n\ndef migration_profile():\n    # A declared smooth scenario profile; not an estimated Rogers-Castro model.\n    return normalized([math.exp(-.5*((a-27)/9)**2)+.20*math.exp(-.5*((a-7)/5)**2)\n                      +.08*math.exp(-.5*((a-62)/8)**2) for a in range(N_AGE)])\n\n\ndef life_e0(hazards):\n    survival=1.; expectation=0.\n    for m in hazards[:100]:\n        expectation += survival * (-math.expm1(-m)/m if m else 1.)\n        survival *= math.exp(-m)\n    return expectation + survival / hazards[100] if hazards[100] > 0 else math.inf\n\n\ndef mortality_from_e0(e0, sex):\n    """Calibrate a model life table; one e0 cannot identify observed age mortality."""\n    if not isinstance(e0,(int,float)) or isinstance(e0,bool):raise ValueError(\'ОПЖ должна быть числом.\')\n    e0=float(e0)\n    if sex not in SEXES or not math.isfinite(e0) or not 30 <= e0 <= 100:\n        raise ValueError(\'ОПЖ для сценарной таблицы должна находиться в интервале 30–100 лет.\')\n    multiplier=1.25 if sex==\'male\' else .85\n    template=[(.004 if sex==\'male\' else .0032) if a==0 else\n        .00013 + multiplier*.000035*math.exp(.092*a) +\n        ((.0003 if sex==\'male\' else .00008)*math.exp(-.5*((a-23)/7)**2)) for a in range(N_AGE)]\n    lo,hi=.005,20.\n    for _ in range(80):\n        mid=(lo+hi)/2\n        actual=life_e0([v*mid for v in template])\n        if actual>e0:lo=mid\n        else:hi=mid\n    hazard=[v*(lo+hi)/2 for v in template]\n    return {\'qx\':[-math.expm1(-m) for m in hazard],\'e0\':life_e0(hazard),\'target_e0\':e0,\n        \'kind\':\'model life table calibrated to e0, NOT observed age-specific mortality\'}\n\n\ndef annual_value(values, year):\n    if not values:\n        raise ValueError(\'Пустая годовая траектория.\')\n    eligible=sorted(int(k) for k in values if int(k)<=year)\n    if not eligible:\n        raise ValueError(f\'Нет значения компоненты на базовый год {year} или ранее.\')\n    v=values[str(eligible[-1])] if str(eligible[-1]) in values else values[eligible[-1]]\n    v=float(v)\n    if not math.isfinite(v):raise ValueError(\'Нечисловой компонент прогноза.\')\n    return v\n\n\ndef validate_input(data):\n    if data.get(\'schema\')!=\'semya.cohort-input/1\':\n        raise ValueError(\'Ожидается схема semya.cohort-input/1.\')\n    start=date.fromisoformat(data[\'base_date\'])\n    scenario=date.fromisoformat(data.get(\'scenario_start\',data[\'base_date\']))\n    if scenario.day!=1 or scenario<start or scenario>=date(2031,1,1):raise ValueError(\'Сценарная дата должна быть первым числом месяца от базы до 2031 года.\')\n    if start.day!=1 or start.year < 2000 or start >= date(2031,1,1):\n        raise ValueError(\'Базовая дата должна быть первым числом месяца до 2031 года, не ранее 2000 года.\')\n    for sex in SEXES:\n        pop=data[\'population\'][sex]\n        if len(pop)!=N_AGE or not all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v) and v>=0 for v in pop) or not math.isfinite(sum(pop)) or sum(pop)<=0:\n            raise ValueError(f\'{sex}: нужны 101 наблюдение по возрастам 0–99 и 100+, без пропусков.\')\n        mort=data[\'mortality\'][sex]\n        if \'qx\' in mort:\n            q=mort[\'qx\']\n            if len(q)!=N_AGE or not all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v) and 0<=v<=1 for v in q):\n                raise ValueError(\'qx: ожидаются 101 вероятность от 0 до 1.\')\n        else:\n            mortality_from_e0(annual_value(mort[\'annual_e0\'],start.year),sex)\n        mig=data[\'migration\'][sex]\n        annual_value(mig[\'annual_net\'],start.year)\n        w=normalized(mig[\'weights\'])\n        if len(w)!=N_AGE:raise ValueError(\'Миграционный профиль должен иметь 101 ячейку.\')\n    for sex in SEXES:\n        for year,value in data[\'migration\'][sex][\'annual_net\'].items():\n            if not str(year).isdigit() or not isinstance(value,(int,float)) or isinstance(value,bool) or not math.isfinite(value):raise ValueError(\'Некорректная годовая миграционная траектория\')\n        for year,value in data[\'mortality\'][sex].get(\'annual_e0\',{}).items():\n            if not str(year).isdigit() or not isinstance(value,(int,float)) or isinstance(value,bool) or not math.isfinite(value) or not 30<=value<=100:raise ValueError(\'Некорректная годовая траектория ОПЖ\')\n    f=data[\'fertility\'];w=normalized(f[\'weights\'])\n    for year,value in f[\'annual_tfr\'].items():\n        if not str(year).isdigit() or not isinstance(value,(int,float)) or isinstance(value,bool) or not math.isfinite(value) or value<0:raise ValueError(\'Некорректная годовая траектория СКР\')\n    if len(w)!=N_AGE or any(x>0 for i,x in enumerate(w) if i<15 or i>49):\n        raise ValueError(\'Возрастные веса рождаемости допустимы только в возрастах 15–49.\')\n    if annual_value(f[\'annual_tfr\'],start.year)<0:raise ValueError(\'СКР не может быть отрицательным.\')\n    for key,value in f.get(\'monthly_tfr\',{}).items():\n        date.fromisoformat(key+\'-01\')\n        if not isinstance(value,(int,float)) or isinstance(value,bool) or not math.isfinite(value) or value<0:raise ValueError(\'Некорректная месячная траектория СКР.\')\n    raw_ratio=data.get(\'sex_ratio\',105.6)\n    if not isinstance(raw_ratio,(int,float)) or isinstance(raw_ratio,bool):raise ValueError(\'Соотношение полов должно быть числом.\')\n    ratio=float(raw_ratio)\n    if not 90<=ratio<=120:raise ValueError(\'Соотношение полов при рождении вне диапазона 90–120.\')\n    return True\n\n\ndef aggregate_age(pop):\n    return [sum(pop[a*12:a*12+12]) for a in range(100)]+[pop[1200]]\n\n\ndef simulate(data, options=None):\n    validate_input(data)\n    opt={\'migration_scale\':1.,\'fertility_scale_end\':1.,\'e0_delta_end\':0.} | (options or {})\n    for k, bounds in {\'migration_scale\':(0,3),\'fertility_scale_end\':(.25,3),\'e0_delta_end\':(-15,15)}.items():\n        v=opt[k]\n        if not isinstance(v,(int,float)) or isinstance(v,bool) or not math.isfinite(v) or not bounds[0]<=v<=bounds[1]:\n            raise ValueError(f\'Параметр {k} вне допустимого диапазона {bounds}.\')\n    start=date.fromisoformat(data[\'base_date\']);start_n=start.year*12+start.month-1;end_n=2030*12+11\n    scenario=date.fromisoformat(data.get(\'scenario_start\',data[\'base_date\']));scenario_n=scenario.year*12+scenario.month-1\n    stocks={s:[v/12 for v in data[\'population\'][s][:100] for _ in range(12)]+[float(data[\'population\'][s][100])] for s in SEXES}\n    fw=normalized(data[\'fertility\'][\'weights\']);mw={s:normalized(data[\'migration\'][s][\'weights\']) for s in SEXES}\n    share_m=data.get(\'sex_ratio\',105.6)/(100+data.get(\'sex_ratio\',105.6))\n    monthly=[];max_balance=0.;cache={}\n    for n in range(start_n,end_n+1):\n        year,month=divmod(n,12);month+=1;key=f\'{year:04d}-{month:02d}\'\n        progress=max(0,(n-scenario_n)/max(1,end_n-scenario_n))\n        tfr=data[\'fertility\'].get(\'monthly_tfr\',{}).get(key)\n        if tfr is None:tfr=annual_value(data[\'fertility\'][\'annual_tfr\'],year)\n        tfr*=1+(opt[\'fertility_scale_end\']-1)*progress\n        before={s:sum(stocks[s]) for s in SEXES};start_f=aggregate_age(stocks[\'female\'])\n        next_stocks={};monthly_survival={};e0_used={}\n        for s in SEXES:\n            spec=data[\'mortality\'][s]\n            if \'qx\' in spec:\n                if opt[\'e0_delta_end\']!=0:\n                    raise ValueError(\'Сдвиг ОПЖ недоступен для заданных напрямую qx. Измените таблицу qx явно.\')\n                qx=spec[\'qx\'];e0_used[s]=None\n            else:\n                e0=annual_value(spec[\'annual_e0\'],year)+opt[\'e0_delta_end\']*progress\n                ck=(s,round(e0,9))\n                if ck not in cache:cache[ck]=mortality_from_e0(e0,s)\n                qx=cache[ck][\'qx\'];e0_used[s]=e0\n            pm=[(1-v)**(1/12) for v in qx];monthly_survival[s]=pm\n            survivors=[v*pm[min(i//12,100)] for i,v in enumerate(stocks[s])]\n            next_stocks[s]=[0.]+survivors[:1199]+[survivors[1199]+survivors[1200]]\n        end_f=aggregate_age(next_stocks[\'female\'])\n        exposure=[(a+b)/2 for a,b in zip(start_f,end_f)]\n        births=sum(exposure[a]*fw[a]*tfr/12 for a in range(N_AGE))\n        deaths={s:before[s]-sum(next_stocks[s]) for s in SEXES};net={}\n        for s,sex_share in [(\'male\',share_m),(\'female\',1-share_m)]:\n            born=births*sex_share;surviving=born*math.sqrt(monthly_survival[s][0])\n            next_stocks[s][0]+=surviving;deaths[s]+=born-surviving\n            net[s]=annual_value(data[\'migration\'][s][\'annual_net\'],year)/12*(opt[\'migration_scale\'] if n>=scenario_n else 1.)\n            for a in range(100):\n                added=net[s]*mw[s][a]/12\n                for j in range(a*12,a*12+12):\n                    next_stocks[s][j]+=added\n                    if next_stocks[s][j]<-1e-8:\n                        raise ValueError(f\'{key}, {s}, возраст {a}: заданный отток превышает численность. Отрицательные ячейки не обрезаются.\')\n            next_stocks[s][1200]+=net[s]*mw[s][100]\n            if next_stocks[s][1200]<-1e-8:raise ValueError(\'Отток превышает численность открытой группы 100+.\')\n        stocks=next_stocks\n        age={s:aggregate_age(stocks[s]) for s in SEXES};total=sum(sum(x) for x in age.values())\n        natural=births-sum(deaths.values());migration=sum(net.values())\n        residual=total-sum(before.values())-natural-migration\n        if not all(math.isfinite(v) for v in [total,births,natural,migration,residual,*deaths.values()]):\n            raise ValueError(\'Переполнение вычислений: проверьте масштабы входных данных и коэффициентов.\')\n        max_balance=max(max_balance,abs(residual))\n        if abs(residual)>max(1e-6,total*1e-10):raise ValueError(\'Нарушен демографический баланс.\')\n        nxt_y,nxt_m=divmod(n+1,12)\n        monthly.append({\'month\':key,\'stock_date\':f\'{nxt_y:04d}-{nxt_m+1:02d}-01\',\n            \'population\':total,\'male\':sum(age[\'male\']),\'female\':sum(age[\'female\']),\n            \'births\':births,\'deaths\':sum(deaths.values()),\'natural_change\':natural,\'net_migration\':migration,\n            \'tfr\':tfr,\'e0_male\':e0_used[\'male\'],\'e0_female\':e0_used[\'female\'],\n            \'children_0_14\':sum(age[\'male\'][:15])+sum(age[\'female\'][:15]),\n            \'age_65_plus\':sum(age[\'male\'][65:])+sum(age[\'female\'][65:]),\n            \'women_15_49\':sum(age[\'female\'][15:50]),\'balance_residual\':residual,\n            \'age\':age})\n    return {\'schema\':\'semya.cohort-result/1\',\'model_version\':VERSION,\'region_id\':data.get(\'region_id\'),\n        \'region_name\':data.get(\'region_name\'),\'base_date\':data[\'base_date\'],\'scenario_start\':data.get(\'scenario_start\',data[\'base_date\']),\'end_date\':END,\n        \'baseline\':data[\'population\'],\'options\':opt,\'months\':monthly,\'max_balance_residual\':max_balance,\n        \'provenance\':data.get(\'provenance\',[]),\'assumptions\':data.get(\'assumptions\',[]),\n        \'interpretation\':\'Условная сценарная передвижка, не официальный и не причинный прогноз. Месяц — вычислительный шаг, а не частота наблюдения всех компонент.\'}\n', 'demography/indicator.py': '"""Monthly diagnostic TFR ensemble with data-conditioned, regularised curvature.\n\nFive model specifications, including harmonic regression and a quasi-periodic GP.\nThe 12-month kernel period is a declared calendar hypothesis, NOT a detected cycle.\nObserved cells are never smoothed, fabricated, interpolated, or mixed with annual rows.\nNew model of the platform; the original May 2026 report remains a separate artefact.\n\nKernel construction: Rasmussen & Williams (2006), Gaussian Processes for Machine Learning,\nchapters 2 and 4; https://gaussianprocess.org/gpml/chapters/.\nFourier terms: Hyndman & Athanasopoulos, Forecasting: Principles and Practice, ch. 10.\n"""\nfrom __future__ import annotations\nimport calendar\nimport hashlib\nimport json\nimport math\nfrom datetime import date\nimport numpy as np\nfrom .indicator_v11 import path as baseline_path\n\nVERSION = \'monthly-nonlinear-ensemble/1.2.0\'\nMODELS = (\'last\', \'local_damped\', \'holt_damped\', \'harmonic_ridge\', \'quasiperiodic_gp\')\nNAMES = {\n    \'last\': \'Последнее значение\',\n    \'local_damped\': \'Локальный затухающий тренд\',\n    \'holt_damped\': \'Затухающий тренд Холта\',\n    \'harmonic_ridge\': \'Тренд + регуляризованная гармоника\',\n    \'quasiperiodic_gp\': \'Тренд + квазипериодический гауссовский процесс\',\n}\nPARAMETERS = {\n    \'minimum_training\': 6, \'validation_horizons\': [1, 3],\n    \'maximum_training_months\': 120, \'maximum_validation_origins\': 24,\n    \'calendar_period_months\': 12, \'minimum_calendar_training\': 12,\n    \'harmonic_ridge_penalty\': 2.0, \'harmonic_anchor_decay_months\': 3.0,\n    \'gp_periodic_length_scale\': 0.9, \'gp_cycle_coherence_months\': 48.0,\n    \'gp_local_length_months\': 2.5, \'gp_local_variance_ratio\': 0.4,\n    \'gp_noise_variance_ratio\': 0.2, \'gp_nugget_variance\': 1e-8,\n    \'gp_mean_estimation\': \'generalized_least_squares\',\n    \'gp_mean_uncertainty\': True,\n    \'minimum_residual_scale\': 1e-4, \'weight_floor_share\': 0.05,\n    \'baseline_specification\': \'monthly-log-ensemble/1.1.1\',\n}\n\n\ndef month_id(value):\n    d = date.fromisoformat(str(value)[:10])\n    return d.year * 12 + d.month - 1\n\n\ndef end_month(n):\n    y, m = divmod(n, 12)\n    m += 1\n    return f\'{y:04d}-{m:02d}-{calendar.monthrange(y, m)[1]:02d}\'\n\n\ndef _linear(y):\n    t = np.arange(len(y), dtype=float) - (len(y) - 1) / 2\n    slope = float(t @ (y - y.mean()) / (t @ t)) if len(y) > 1 else 0.0\n    return float(y.mean()), slope, t\n\n\ndef _model(y, horizon, method):\n    """Return the conditional log path and its additive components; no future data."""\n    y = np.asarray(y[-PARAMETERS[\'maximum_training_months\']:], dtype=float)\n    n = len(y)\n    future = np.arange(n, n + horizon, dtype=float) - (n - 1) / 2\n    steps = np.arange(1, horizon + 1, dtype=float)\n    zero = np.zeros(horizon)\n    active = n >= PARAMETERS[\'minimum_calendar_training\']\n    if method in MODELS[:3]:\n        p = np.asarray(baseline_path(y.tolist(), horizon, method))\n        return dict(path=p, trend=p, cycle=zero, local=zero, variance=zero,\n                    fit=None, meta={\'calendar_active\': False, \'training_months\': n})\n    level, slope, t = _linear(y)\n    trend = level + slope * future\n    if method == \'harmonic_ridge\':\n        if not active:\n            # A fully specified fallback: no harmonic is estimated on a partial year.\n            p = y[-1] + slope * steps\n            return dict(path=p, trend=p, cycle=zero, local=zero, variance=zero,\n                        fit=level + slope * t,\n                        meta={\'calendar_active\': False, \'training_months\': n,\n                              \'fallback\': \'log-linear path anchored at last observation\'})\n        def design(x):\n            phase = 2 * np.pi * x / PARAMETERS[\'calendar_period_months\']\n            return np.column_stack([np.ones(len(x)), x, np.sin(phase), np.cos(phase)])\n        X = design(t)\n        penalty = np.diag([0., 0., PARAMETERS[\'harmonic_ridge_penalty\'],\n                           PARAMETERS[\'harmonic_ridge_penalty\']])\n        beta = np.linalg.solve(X.T @ X + penalty, X.T @ y)\n        fit = X @ beta\n        A = design(future)\n        trend = A[:, :2] @ beta[:2]\n        cycle = A[:, 2:] @ beta[2:]\n        local = (y[-1] - fit[-1]) * np.exp(-steps / PARAMETERS[\'harmonic_anchor_decay_months\'])\n        return dict(path=trend + cycle + local, trend=trend, cycle=cycle, local=local,\n                    variance=zero, fit=fit,\n                    meta={\'calendar_active\': True, \'training_months\': n,\n                          \'coefficients_log\': beta.tolist(),\n                          \'amplitude_log\': float(np.hypot(beta[2], beta[3])),\n                          \'period_months\': PARAMETERS[\'calendar_period_months\'],\n                          \'period_status\': \'assumed, not estimated\'})\n    if method != \'quasiperiodic_gp\':\n        raise ValueError(\'Неизвестная модель: \' + method)\n    residual = y - (level + slope * t)\n    scale = max(float(np.std(residual)), PARAMETERS[\'minimum_residual_scale\'])\n\n    def kernels(x, z):\n        delta = np.subtract.outer(x, z)\n        period = PARAMETERS[\'calendar_period_months\']\n        cyc = np.exp(-2 * np.sin(np.pi * delta / period) ** 2 /\n                     PARAMETERS[\'gp_periodic_length_scale\'] ** 2)\n        cyc *= np.exp(-0.5 * (delta / PARAMETERS[\'gp_cycle_coherence_months\']) ** 2)\n        if not active:\n            cyc[:] = 0\n        local = PARAMETERS[\'gp_local_variance_ratio\'] * np.exp(\n            -0.5 * (delta / PARAMETERS[\'gp_local_length_months\']) ** 2)\n        return scale ** 2 * cyc, scale ** 2 * local\n\n    Kc, Kl = kernels(t, t)\n    noise = scale ** 2 * PARAMETERS[\'gp_noise_variance_ratio\'] + PARAMETERS[\'gp_nugget_variance\']\n    L = np.linalg.cholesky(Kc + Kl + np.eye(n) * noise)\n    # Jointly estimate the linear mean with the covariance, so a calendar pattern\n    # is not mechanically absorbed into the slope of the extrapolated trend.\n    A = np.column_stack([np.ones(n), t])\n    Af = np.column_stack([np.ones(horizon), future])\n    invA = np.linalg.solve(L.T, np.linalg.solve(L, A))\n    invY = np.linalg.solve(L.T, np.linalg.solve(L, y))\n    precision_beta = A.T @ invA\n    beta = np.linalg.solve(precision_beta, A.T @ invY)\n    level, slope = map(float, beta)\n    trend = Af @ beta\n    residual = y - A @ beta\n    alpha = np.linalg.solve(L.T, np.linalg.solve(L, residual))\n    Fc, Fl = kernels(future, t)\n    cycle = Fc @ alpha\n    local = Fl @ alpha\n    fit = level + slope * t + (Kc + Kl) @ alpha\n    # Smooth endpoint correction is estimated from the final residual, never drawn by hand.\n    local += (y[-1] - fit[-1]) * np.exp(-steps / PARAMETERS[\'gp_local_length_months\'])\n    solved = np.linalg.solve(L, (Fc + Fl).T)\n    prior_diag = scale ** 2 * (int(active) + PARAMETERS[\'gp_local_variance_ratio\'])\n    mean_design_residual = Af - (Fc + Fl) @ invA\n    mean_variance = np.einsum(\'ij,ji->i\', mean_design_residual,\n                             np.linalg.solve(precision_beta, mean_design_residual.T))\n    variance = np.maximum(0., prior_diag - (solved ** 2).sum(axis=0) + mean_variance)\n    return dict(path=trend + cycle + local, trend=trend, cycle=cycle, local=local,\n                variance=variance, fit=fit,\n                meta={\'calendar_active\': active, \'training_months\': n, \'level_log\': level,\n                      \'trend_per_month_log\': slope, \'residual_scale_log\': scale,\n                      \'period_months\': PARAMETERS[\'calendar_period_months\'],\n                      \'period_status\': \'assumed, not estimated\',\n                      \'cycle_coherence_months\': PARAMETERS[\'gp_cycle_coherence_months\'],\n                      \'mean_estimation\': PARAMETERS[\'gp_mean_estimation\']})\n\n\ndef path(y, horizon, method):\n    """Compatibility helper for tests and isolated model inspection."""\n    return _model(y, horizon, method)[\'path\'].tolist()\n\n\ndef _errors_to_weights(errors):\n    mse = np.array([np.mean(np.square(errors[m])) if errors[m] else np.nan for m in MODELS])\n    finite = mse[np.isfinite(mse)]\n    floor = max(1e-8, float(finite.mean()) * PARAMETERS[\'weight_floor_share\']) if len(finite) else 1e-8\n    raw = np.array([1 / (v + floor) if math.isfinite(v) else 1. for v in mse])\n    return raw / raw.sum(), mse\n\n\ndef _validation(y, dates):\n    """Rolling-origin validation of fixed model algorithms and prequential ensemble.\n\n    At an ensemble origin only errors whose target was observed strictly earlier may\n    enter its weights. Endpoint normalisation, regression and kernels are refit inside\n    each origin. No full-series residuals or future targets enter these fits.\n    """\n    backtests = []\n    outer = []\n    first = max(PARAMETERS[\'minimum_training\'], len(y) - PARAMETERS[\'maximum_validation_origins\'])\n    for origin in range(first, len(y)):\n        H = min(max(PARAMETERS[\'validation_horizons\']), len(y) - origin)\n        before = {m: [] for m in MODELS}\n        for b in backtests:\n            if b[\'target_index\'] < origin:\n                before[b[\'model\']].append(b[\'log_error\'])\n        pre_weights, _ = _errors_to_weights(before)\n        predictions = {m: _model(y[:origin], H, m) for m in MODELS}\n        for horizon in PARAMETERS[\'validation_horizons\']:\n            target = origin + horizon - 1\n            if target >= len(y):\n                continue\n            logs = np.array([predictions[m][\'path\'][horizon - 1] for m in MODELS])\n            for i, m in enumerate(MODELS):\n                predicted = float(np.exp(logs[i]))\n                if not math.isfinite(predicted):\n                    raise ValueError(\'Модель нестабильна при временной проверке.\')\n                backtests.append({\'model\': m, \'train_end\': dates[origin - 1],\n                                  \'target\': dates[target], \'target_index\': target,\n                                  \'horizon\': horizon, \'observed\': float(np.exp(y[target])),\n                                  \'predicted\': predicted, \'log_error\': float(logs[i] - y[target]),\n                                  \'calendar_active\': predictions[m][\'meta\'][\'calendar_active\']})\n            if origin > first:\n                pred = float(np.exp(pre_weights @ logs))\n                outer.append({\'train_end\': dates[origin - 1], \'target\': dates[target],\n                              \'horizon\': horizon, \'observed\': float(np.exp(y[target])),\n                              \'predicted\': pred, \'last_value_prediction\': float(np.exp(y[origin - 1])),\n                              \'weight_errors_end\': max((b[\'target\'] for b in backtests if b[\'target_index\'] < origin), default=None),\n                              \'weights\': {m: float(v) for m, v in zip(MODELS, pre_weights)}})\n    return backtests, outer\n\n\ndef forecast(observations, end=\'2030-12-31\'):\n    obs = sorted(observations, key=lambda x: x[\'date\'])\n    if len(obs) < 6:\n        raise ValueError(\'Для прогнозирования нужны хотя бы 6 последовательных месячных наблюдений.\')\n    ids = [month_id(o[\'date\']) for o in obs]\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Дубли месячных наблюдений.\')\n    if any(b - a != 1 for a, b in zip(ids, ids[1:])):\n        raise ValueError(\'Разрыв месячной сетки: заполнение или сжатие времени запрещено.\')\n    if any(not isinstance(o[\'value\'], (int, float)) or isinstance(o[\'value\'], bool) or\n           not math.isfinite(o[\'value\']) or o[\'value\'] <= 0 for o in obs):\n        raise ValueError(\'Нужны положительные конечные значения СКР без пропусков.\')\n    H = max(0, month_id(end) - ids[-1])\n    y = np.log([o[\'value\'] for o in obs])\n    backtests, outer = _validation(y, [o[\'date\'] for o in obs])\n    errors = {m: [b[\'log_error\'] for b in backtests if b[\'model\'] == m] for m in MODELS}\n    weights, mse = _errors_to_weights(errors)\n    fitted = {m: _model(y, H, m) for m in MODELS}\n    one_step = [math.log(b[\'predicted\'] / b[\'observed\']) for b in outer if b[\'horizon\'] == 1]\n    if not one_step:\n        one_step = [b[\'log_error\'] for b in backtests if b[\'horizon\'] == 1]\n    sigma = float(np.sqrt(np.mean(np.square(one_step)))) if one_step else float(np.std(np.diff(y)))\n    rows = []\n    for i in range(H):\n        vals = np.array([fitted[m][\'path\'][i] for m in MODELS])\n        mu = float(weights @ vals)\n        spread = float(weights @ ((vals - mu) ** 2))\n        gp_var = float(sum(weights[j] ** 2 * fitted[m][\'variance\'][i] for j, m in enumerate(MODELS)))\n        se = math.sqrt((i + 1) * sigma ** 2 + spread + gp_var)\n        if max(abs(mu), abs(mu + 1.96 * se), abs(mu - 1.96 * se)) > 25:\n            raise ValueError(\'Модель вышла за численно устойчивый диапазон; результат не публикуется.\')\n        trend, cycle, local = [float(sum(weights[j] * fitted[m][key][i] for j, m in enumerate(MODELS)))\n                               for key in [\'trend\', \'cycle\', \'local\']]\n        rows.append({\'date\': end_month(ids[-1] + i + 1), \'value\': math.exp(mu),\n                     \'lo80\': math.exp(mu - 1.2815515655 * se), \'hi80\': math.exp(mu + 1.2815515655 * se),\n                     \'lo95\': math.exp(mu - 1.9599639845 * se), \'hi95\': math.exp(mu + 1.9599639845 * se),\n                     \'model_min\': float(np.exp(vals).min()), \'model_max\': float(np.exp(vals).max()),\n                     \'members\': {m: float(math.exp(vals[j])) for j, m in enumerate(MODELS)},\n                     \'trend_value\': math.exp(trend), \'cycle_factor\': math.exp(cycle),\n                     \'local_factor\': math.exp(local), \'cycle_percent\': math.expm1(cycle) * 100,\n                     \'prediction_log_sd\': se})\n    parameters = dict(PARAMETERS)\n    fingerprint = hashlib.sha256(json.dumps({\'obs\': obs, \'end\': end, \'version\': VERSION,\n                                           \'parameters\': parameters}, ensure_ascii=False,\n                                           sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n    models = []\n    for j, m in enumerate(MODELS):\n        b = [b for b in backtests if b[\'model\'] == m]\n        models.append({\'id\': m, \'name\': NAMES[m], \'weight\': float(weights[j]), \'n_tests\': len(b),\n                       \'log_rmse\': math.sqrt(mse[j]) if math.isfinite(mse[j]) else None,\n                       \'mae\': float(np.mean([abs(bi[\'predicted\'] - bi[\'observed\']) for bi in b])) if b else None,\n                       \'calendar_tests\': sum(bi[\'calendar_active\'] for bi in b),\n                       \'fit\': fitted[m][\'meta\']})\n    mae = lambda field: float(np.mean([abs(b[field] - b[\'observed\']) for b in outer])) if outer else None\n    return {\'schema\': \'semya.indicator-projection/1\', \'model_version\': VERSION,\n            \'input_sha256\': fingerprint, \'source_as_of\': obs[-1][\'date\'], \'end_date\': end,\n            \'n_observations\': len(obs), \'observations\': obs, \'forecast\': rows, \'models\': models,\n            \'backtest\': [{k: v for k, v in b.items() if k != \'target_index\'} for b in backtests],\n            \'ensemble_backtest\': outer,\n            \'validation\': {\'kind\': \'prequential, expanding origin; weights use past targets only\',\n                           \'n_tests\': len(outer), \'mae\': mae(\'predicted\'),\n                           \'last_value_mae\': mae(\'last_value_prediction\'), \'max_test_horizon\': 3,\n                           \'historical_vintages_available\': False},\n            \'parameters\': parameters,\n            \'intervals\': {\'type\': \'conditional_log_normal_heuristic\',\n                          \'innovation_log_rmse\': sigma, \'calibrated\': False},\n            \'curvature\': {\'kind\': \'estimated Fourier/GP residuals, not chart smoothing\',\n                          \'calendar_hypothesis_months\': 12,\n                          \'calendar_estimated\': len(obs) >= 12,\n                          \'period_estimated\': False, \'seasonality_confirmed\': False,\n                          \'training_cycles\': len(obs) / 12},\n            \'caveats\': [\n                \'Новая модель платформы; не заменяет сохранённый прогноз исходной экспертизы.\',\n                \'Помесячно опубликованный СКР — годовой по размерности коэффициент, не число детей за месяц.\',\n                \'Годовые и накопительные строки не входят в месячное обучение. Пропуски не интерполируются.\',\n                \'Период 12 месяцев задан как проверяемая календарная гипотеза. Один цикл не доказывает сезонность. Соседние оперативные оценки могут быть зависимы.\',\n                \'Параметры гармоники и гауссовского процесса оцениваются по исходному ряду; наблюдения не сглаживаются. На ровном ряду искусственные колебания не создаются.\',\n                \'Временные проверки охватывают 1 и 3 месяца. Надёжность прогноза до 2030 года ими не подтверждена; используются текущие версии прошлых значений, не архивы публикаций.\',\n                \'80/95%-полосы — условная оценка: ошибки краткосрочной проверки, расхождение моделей и остаточная неопределённость GP. Фактическое покрытие и структурные шоки не проверены.\',\n                \'Национальный ряд рассчитан независимо; региональные коэффициенты не усредняются в СКР России.\',\n            ]}\n', 'demography/indicator_v11.py': '"""Explicit monthly log-scale diagnostic ensemble (new implementation, not the 2026 report model).\nNo annual/cumulative rows, interpolation of gaps, seasonal inference from one year,\npolicy coefficients, cross-region averaging or hidden fallback observations.\n"""\nfrom __future__ import annotations\nimport calendar, hashlib, json, math, statistics\nfrom datetime import date\nVERSION = \'monthly-log-ensemble/1.1.1\'\nMODELS = (\'last\', \'local_damped\', \'holt_damped\')\nNAMES = {\'last\':\'Последнее значение\', \'local_damped\':\'Локальный затухающий тренд\', \'holt_damped\':\'Затухающий тренд Холта\'}\nPARAMETERS = {\'local_window\':12,\'phi\':0.96,\'alpha\':0.5,\'beta\':0.1,\'minimum_training\':6,\'validation_horizons\':[1,3]}\n\ndef month_id(value):\n    d=date.fromisoformat(str(value)[:10]); return d.year*12+d.month-1\n\ndef end_month(n):\n    y,m=divmod(n,12); m+=1; return f\'{y:04d}-{m:02d}-{calendar.monthrange(y,m)[1]:02d}\'\n\ndef mean(a): return sum(a)/len(a)\n\ndef path(y,horizon,method):\n    phi=PARAMETERS[\'phi\']; n=len(y)\n    if method==\'last\': return [y[-1]]*horizon\n    if method==\'local_damped\':\n        a=y[-min(PARAMETERS[\'local_window\'],n):]; center=(len(a)-1)/2\n        slope=sum((i-center)*(v-mean(a)) for i,v in enumerate(a))/sum((i-center)**2 for i in range(len(a))) if len(a)>1 else 0\n        level=y[-1]\n    elif method==\'holt_damped\':\n        level=y[0]; slope=(y[min(2,n-1)]-y[0])/min(2,n-1) if n>1 else 0\n        alpha,beta=PARAMETERS[\'alpha\'],PARAMETERS[\'beta\']\n        for v in y[1:]:\n            previous=level; level=alpha*v+(1-alpha)*(level+phi*slope)\n            slope=beta*(level-previous)+(1-beta)*phi*slope\n    else: raise ValueError(\'Unknown model\')\n    return [level+slope*phi*(1-phi**k)/(1-phi) for k in range(1,horizon+1)]\n\ndef forecast(observations, end=\'2030-12-31\'):\n    obs=sorted(observations,key=lambda x:x[\'date\'])\n    if len(obs)<6: raise ValueError(\'Для прогнозирования нужны хотя бы 6 последовательных месячных наблюдений.\')\n    ids=[month_id(o[\'date\']) for o in obs]\n    if len(set(ids))!=len(ids): raise ValueError(\'Дубли месячных наблюдений.\')\n    # A new gap cannot silently be interpolated or compressed into one step.\n    if any(b-a!=1 for a,b in zip(ids,ids[1:])): raise ValueError(\'Разрыв месячной сетки: заполнение или сжатие времени запрещено.\')\n    if any(not isinstance(o[\'value\'],(int,float)) or isinstance(o[\'value\'],bool) or not math.isfinite(o[\'value\']) or o[\'value\']<=0 for o in obs):\n        raise ValueError(\'Нужны положительные конечные значения СКР без пропусков.\')\n    H=max(0,month_id(end)-ids[-1]);y=[math.log(o[\'value\']) for o in obs]\n    backtests=[];errors={m:[] for m in MODELS}\n    for origin in range(6,len(y)):\n        for horizon in PARAMETERS[\'validation_horizons\']:\n            target=origin+horizon-1\n            if target>=len(y):continue\n            for m in MODELS:\n                pred=path(y[:origin],horizon,m)[-1];err=pred-y[target]\n                errors[m].append(err)\n                backtests.append({\'model\':m,\'train_end\':obs[origin-1][\'date\'],\'target\':obs[target][\'date\'],\'horizon\':horizon,\'observed\':obs[target][\'value\'],\'predicted\':math.exp(pred),\'log_error\':err})\n    mse={m:mean([e*e for e in errors[m]]) if errors[m] else None for m in MODELS}\n    # A small floor prevents a single accidentally perfect short validation from monopolising weights.\n    floor=max(1e-8,mean([v for v in mse.values() if v is not None])*0.05) if backtests else 1e-8\n    raw={m:1/(mse[m]+floor) if mse[m] is not None else 1 for m in MODELS};total=sum(raw.values());weights={m:raw[m]/total for m in MODELS}\n    paths={m:path(y,H,m) for m in MODELS}\n    # Conditional, deliberately labelled uncalibrated bands. No causal/policy probability is inferred.\n    one_step=[b[\'log_error\'] for b in backtests if b[\'horizon\']==1]\n    sigma=math.sqrt(mean([e*e for e in one_step])) if one_step else statistics.pstdev([b-a for a,b in zip(y,y[1:])])\n    result=[]\n    for i in range(H):\n        mu=sum(weights[m]*paths[m][i] for m in MODELS)\n        spread=sum(weights[m]*(paths[m][i]-mu)**2 for m in MODELS)\n        se=math.sqrt((i+1)*sigma*sigma+spread)\n        if max(abs(mu),abs(mu+1.96*se),abs(mu-1.96*se))>25: raise ValueError(\'Модель вышла за численно устойчивый диапазон; результат не публикуется.\')\n        row={\'date\':end_month(ids[-1]+i+1),\'value\':math.exp(mu),\'lo80\':math.exp(mu-1.2815515655*se),\'hi80\':math.exp(mu+1.2815515655*se),\'lo95\':math.exp(mu-1.9599639845*se),\'hi95\':math.exp(mu+1.9599639845*se),\'model_min\':min(math.exp(paths[m][i]) for m in MODELS),\'model_max\':max(math.exp(paths[m][i]) for m in MODELS)}\n        row[\'members\']={m:math.exp(paths[m][i]) for m in MODELS};result.append(row)\n    fingerprint=hashlib.sha256(json.dumps({\'obs\':obs,\'end\':end,\'version\':VERSION,\'parameters\':PARAMETERS},ensure_ascii=False,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n    models=[]\n    for m in MODELS:\n        b=[b for b in backtests if b[\'model\']==m]\n        models.append({\'id\':m,\'name\':NAMES[m],\'weight\':weights[m],\'n_tests\':len(b),\'log_rmse\':math.sqrt(mse[m]) if mse[m] is not None else None,\'mae\':mean([abs(bi[\'predicted\']-bi[\'observed\']) for bi in b]) if b else None})\n    return {\'schema\':\'semya.indicator-projection/1\',\'model_version\':VERSION,\'input_sha256\':fingerprint,\'source_as_of\':obs[-1][\'date\'],\'end_date\':end,\'n_observations\':len(obs),\'observations\':obs,\'forecast\':result,\'models\':models,\'backtest\':backtests,\'parameters\':PARAMETERS,\'intervals\':{\'type\':\'conditional_log_normal\',\'innovation_log_rmse\':sigma,\'calibrated\':False},\'caveats\':[\'Новая модель платформы; не заменяет сохранённый прогноз исходной экспертизы.\',\'Помесячно опубликованный СКР является годовым по размерности коэффициентом, а не числом детей за отдельный месяц.\',\'Годовые и накопительные строки не включены в месячное обучение.\',\'Короткая история и зависимость соседних оценок ограничивают надёжность экстраполяции. Сезонность не оценивается.\',\'Условные 80/95%-полосы построены из ошибок краткосрочной проверки с увеличением дисперсии по горизонту. Фактическое покрытие не валидировано; структурные шоки не учтены.\',\'Национальная серия рассчитана независимо. Региональные коэффициенты не усредняются в показатель России.\']}\n', 'demography/__init__.py': '', 'reproduce_projection.py': '#!/usr/bin/env python3\n"""Reproduce a downloaded individual forecast or cohort package using Python 3.11+."""\nimport argparse,json,math\nfrom pathlib import Path\nfrom demography.indicator import forecast, VERSION\nfrom demography.indicator_v11 import forecast as legacy_forecast, VERSION as LEGACY_VERSION\nfrom demography.cohort import simulate\n\ndef same(a,b,path=\'root\'):\n    if isinstance(a,(int,float)) and not isinstance(a,bool):\n        if not isinstance(b,(int,float)) or not math.isclose(a,b,rel_tol=2e-10,abs_tol=2e-6):raise ValueError(f\'Несовпадение {path}: {a} ≠ {b}\')\n    elif isinstance(a,list):\n        if len(a)!=len(b):raise ValueError(\'Несовпадение длины \'+path)\n        for i,(x,y) in enumerate(zip(a,b)):same(x,y,path+f\'[{i}]\')\n    elif isinstance(a,dict):\n        for k,v in a.items():\n            if k not in b:raise ValueError(\'Нет поля \'+path+\'.\'+k)\n            same(v,b[k],path+\'.\'+k)\n    elif a!=b:raise ValueError(\'Несовпадение \'+path)\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'kind\',choices=[\'indicator\',\'cohort\']);p.add_argument(\'file\',type=Path);p.add_argument(\'--output\',type=Path);args=p.parse_args()\n    saved=json.loads(args.file.read_text(encoding=\'utf-8-sig\'))\n    if args.kind==\'indicator\':\n        version=saved.get(\'model_version\')\n        if version == VERSION: runner=forecast\n        elif version == LEGACY_VERSION: runner=legacy_forecast\n        else: raise ValueError(\'Неподдерживаемая версия модели: \'+str(version))\n        fresh=runner(saved[\'observations\'],saved[\'end_date\'])\n        if saved.get(\'model_version\')!=fresh[\'model_version\']:raise ValueError(\'Версия модели изменилась; используйте код из того же релиза, что и JSON.\')\n        for key in [\'forecast\',\'models\',\'backtest\',\'input_sha256\',\'ensemble_backtest\',\'parameters\',\'validation\',\'curvature\']:\n            if key in fresh: same(fresh[key],saved[key],key)\n        print(\'Совпали траектории, параметры весов, проверки и хеш входа.\')\n    else:\n        data=saved.get(\'input\',saved);fresh=simulate(data,saved.get(\'options\',{}));old=saved.get(\'result\')\n        if old:\n            if old.get(\'model_version\')!=fresh[\'model_version\']:raise ValueError(\'Версия алгоритма изменилась; нужен код соответствующего релиза.\')\n            same(fresh[\'months\'],old[\'months\'],\'months\');same(fresh[\'max_balance_residual\'],old[\'max_balance_residual\'],\'balance\')\n            print(\'Проверены все месяцы, возрастные ячейки и демографический баланс.\')\n        else:print(\'Входной файл рассчитан; эталонный результат не приложен.\')\n    if args.output:args.output.write_text(json.dumps(fresh,ensure_ascii=False,allow_nan=False,indent=2),encoding=\'utf-8\')\nif __name__==\'__main__\':main()\n'}
for name, text in FILES.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
print("Алгоритмы подготовлены")


## Загрузка сохранённого расчёта

В Google Colab выберите файл `forecast_…json` или `cohort_…json`. Для запуска вне Colab укажите локальный путь.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
except ImportError:
    filename = input("Путь к JSON: ").strip()
obj = json.loads(Path(filename).read_text(encoding="utf-8-sig"))
schema = obj.get("schema", "")
if schema == "semya.indicator-projection/1":
    kind = "indicator"
elif schema in {"semya.cohort-package/1", "semya.cohort-input/1"}:
    kind = "cohort"
else:
    raise ValueError("Неизвестная схема файла: " + schema)
print("Тип расчёта:", kind)


In [ ]:
result = subprocess.run([sys.executable, "reproduce_projection.py", kind, filename, "--output", "reproduced.json"], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr)
print("Расчёт сохранён в reproduced.json")


In [ ]:
try:
    from google.colab import files
    files.download("reproduced.json")
except ImportError:
    print(Path("reproduced.json").resolve())


## Ограничения

Условные интервалы ансамбля не являются подтверждёнными частотными интервалами и не дают вероятности воздействия нацпроекта. Передвижка зависит от наблюдаемой базы и заданных возрастных/временных предпосылок. Модельная таблица смертности из ОПЖ не равна наблюдаемой возрастной таблице. Месячный шаг не означает ежемесячное наблюдение всех компонент.